# Ask AOPWiki with a local Qwen model

This is the same RDF/MCP workflow as `notebooks/mcp/01_ask_aopwiki.ipynb`, using the Qwen checkpoint served by llama.cpp on the HPC H100. The model endpoint is OpenAI-compatible, so the package's PydanticAI agent and validation remain unchanged.

Set `QWEN_BASE_URL` and `QWEN_MODEL` to compare another installed Qwen checkpoint.

In [ ]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv
from mcp import Client as MCPClient, StdioServerParameters
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider
from pydantic_ai.usage import UsageLimits
from rdfsolve.pydantic_ai import ask, save_answer
from rdfsolve.mcp import query_answer, read_plan
from IPython.display import Markdown, display

load_dotenv('../.env')
question = os.getenv('RDFSOLVE_QUESTION', 'Which Adverse Outcome Pathways represent thyroid issues in humans or mammals, and which genes are described to be related to these pathways? What method was used to annotate these gene relationships?')
base_url = os.getenv('QWEN_BASE_URL', 'http://127.0.0.1:8080/v1')
model_name = os.getenv('QWEN_MODEL', 'qwen36-35b-a3b')
model = OpenAIChatModel(model_name, provider=OpenAIProvider(base_url=base_url, api_key='not-needed'))
print(f'Model: {model_name}; endpoint: {base_url}')

In [ ]:
server = MCPClient(
    StdioServerParameters(
        command=sys.executable,
        args=['-m', 'rdfsolve.mcp', '--schema', '../data/aopwikirdf.schema.json',
              '--source-id', 'aopwikirdf', '--log', 'output/qwen-investigation-session.json'],
    ),
    read_timeout_seconds=900,
)

In [ ]:
async with server:
    answer = await ask(
        server, question, model=model,
        model_settings={'max_tokens': 2000, 'temperature': 0},
        usage_limits=UsageLimits(request_limit=12, tool_calls_limit=18, total_tokens_limit=60000),
    )
    display(Markdown(answer.output.text))
    usage = answer.usage
    references = [item.reference for item in answer.output.results]
    result = await query_answer(server, references, name='Pathways, genes and annotation methods') if references else None
    investigation = await read_plan(server)
save_answer(answer, 'output/qwen-investigation-session.json', question=question)
if result is None:
    raise ValueError('The Qwen agent did not produce an answer query; see output/qwen-investigation-session.json')
display(Markdown('## Retrieved result'))
display(result.table())
print(f'Rows: {len(result.table())}; model usage: {usage}')

In [ ]:
from pathlib import Path
from rdflib import Graph
from rdfsolve.query_log import QueryLog

Path('output').mkdir(exist_ok=True)
result.to_shacl().serialize('output/qwen-answer.shacl.ttl', format='turtle')
subset = Graph()
for record in result.records():
    subset += record.to_graph()
subset.serialize('output/qwen-answer.ttl', format='turtle')
print(result.query)
display(QueryLog.read('output/qwen-investigation-session.json').tools())